# Protein MD Setup tutorial using BioExcel Building Blocks (biobb)
**Based on the official GROMACS tutorial:** [http://www.mdtutorials.com/gmx/lysozyme/index.html](http://www.mdtutorials.com/gmx/lysozyme/index.html)
***
This tutorial aims to illustrate the process of **setting up a simulation system** containing a **protein**, step by step, using the **BioExcel Building Blocks library (biobb)**. The particular example used is the **Lysozyme** protein (PDB code 1AKI, [https://doi.org/10.2210/pdb1AKI/pdb](https://doi.org/10.2210/pdb1AKI/pdb)). 
***

## Settings

### Biobb modules used

 - [biobb_io](https://github.com/bioexcel/biobb_io): Tools to fetch biomolecular data from public databases.
 - [biobb_model](https://github.com/bioexcel/biobb_model): Tools to model macromolecular structures.
 - [biobb_gromacs](https://github.com/bioexcel/biobb_gromacs): Tools to setup and run Molecular Dynamics simulations.
 - [biobb_analysis](https://github.com/bioexcel/biobb_analysis): Tools to analyse Molecular Dynamics trajectories.
 
### Auxiliary libraries used

* [jupyter](https://jupyter.org/): Free software, open standards, and web services for interactive computing across all programming languages.
* [nglview](http://nglviewer.org/#nglview): Jupyter/IPython widget to interactively view molecular structures and trajectories in notebooks.
* [plotly](https://plot.ly/python/offline/): Python interactive graphing library integrated in Jupyter notebooks.
* [simpletraj](https://github.com/arose/simpletraj): Lightweight coordinate-only trajectory reader based on code from GROMACS, MDAnalysis and VMD.

### Conda Installation and Launch

```console
git clone https://github.com/bioexcel/biobb_wf_md_setup.git
cd biobb_wf_md_setup
conda env create -f conda_env/environment.yml
conda activate biobb_wf_md_setup
jupyter-notebook biobb_wf_md_setup/notebooks/biobb_wf_md_setup.ipynb
```

***
## Pipeline steps
 1. [Input Parameters](#input)
 2. [Fetching PDB Structure](#fetch)
 3. [Fix Protein Structure](#fix)
 4. [Create Protein System Topology](#top)
 5. [Create Solvent Box](#box)
 6. [Fill the Box with Water Molecules](#water)
 7. [Adding Ions](#ions)
 8. [Energetically Minimize the System](#min)
 9. [Equilibrate the System (NVT)](#nvt)
 10. [Equilibrate the System (NPT)](#npt)
 11. [Free Molecular Dynamics Simulation](#free)
 12. [Post-processing and Visualizing Resulting 3D Trajectory](#post)
 13. [Output Files](#output)
 14. [Questions & Comments](#questions)
 
***
<img src="https://bioexcel.eu/wp-content/uploads/2019/04/Bioexcell_logo_1080px_transp.png" alt="Bioexcel2 logo"
	title="Bioexcel2 logo" width="400" />
***


## Initializing colab
The cell below is used only in case this notebook is executed via **Google Colab**. This process can take a **few minutes** since **miniforge** is **downloaded** and **installed** first, and then the **conda environment** must be created.

In [1]:
# Only executed when using google colab
import sys
import os
# if 'google.colab' in sys.modules:
#   !git clone https://github.com/bioexcel/biobb_wf_md_setup.git
#   !wget https://github.com/conda-forge/miniforge/releases/download/26.1.0-0/Miniforge3-26.1.0-0-Linux-x86_64.sh
#   !bash Miniforge3-26.1.0-0-Linux-x86_64.sh -b -p /usr/local/miniforge
#   !/usr/local/miniforge/condabin/mamba create -f biobb_wf_md_setup/conda_env/environment.yml -y
#   %env PATH=/usr/local/bin:/opt/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin:/usr/local/miniforge/envs/biobb_wf_md_setup/bin
#   _ = (sys.path.append("/usr/local/miniforge/envs/biobb_wf_md_setup/lib/python3.12/site-packages/"))
#   from google.colab import output
#   output.enable_custom_widget_manager()
#   os.chdir('/content/biobb_wf_md_setup/biobb_wf_md_setup/notebooks')

<a id="input"></a>
## Input parameters
**Input parameters** needed:
 - **pdbCode**: PDB code of the protein structure (e.g. 1AKI, [https://doi.org/10.2210/pdb1AKI/pdb](https://doi.org/10.2210/pdb1AKI/pdb))

<a id="fetch"></a>
***
## Fetching PDB structure
Downloading **PDB structure** with the **protein molecule** from the RCSB PDB database.<br>
Alternatively, a **PDB file** can be used as starting structure. <br>
***
**Building Blocks** used:
 - [Pdb](https://biobb-io.readthedocs.io/en/latest/api.html#module-api.pdb) from **biobb_io.api.pdb**
***

In [2]:
# Downloading desired PDB file 
# Import module
import nglview
import ipywidgets
import sys

# pdbCode = "1AKI"
from biobb_io.api.pdb import pdb

# Create properties dict and inputs/outputs
peptide_pdb = 'peptide.pdb'
# prop = {
#     'pdb_code': pdbCode
# }

# #Create and launch bb
# pdb(output_pdb_path=downloaded_pdb,
#     properties=prop)

<a id="vis3D"></a>
### Visualizing 3D structure
Visualizing the downloaded/given **PDB structure** using **NGL**:    

In [3]:
# Show protein
view = nglview.show_structure_file(peptide_pdb)
view.add_representation(repr_type='ball+stick', selection='all')
view._remote_call('setSize', target='Widget', args=['','600px'])
view

NGLWidget()

<a id="fix"></a>
***
## Fix protein structure
**Checking** and **fixing** (if needed) the protein structure:<br>
- **Modeling** **missing side-chain atoms**, modifying incorrect **amide assignments**, choosing **alternative locations**.<br>
- **Checking** for missing **backbone atoms**, **heteroatoms**, **modified residues** and possible **atomic clashes**.

***
**Building Blocks** used:
 - [FixSideChain](https://biobb-model.readthedocs.io/en/latest/model.html#module-model.fix_side_chain) from **biobb_model.model.fix_side_chain**
***

In [4]:
# # Check & Fix PDB
# # Import module
# from biobb_model.model.fix_side_chain import fix_side_chain

# # Create prop dict and inputs/outputs
# fixed_pdb = pdbCode + '_fixed.pdb'

# # Create and launch bb
# fix_side_chain(input_pdb_path=downloaded_pdb, 
#              output_pdb_path=fixed_pdb)

### Visualizing 3D structure
Visualizing the fixed **PDB structure** using **NGL**. In this particular example, the checking step didn't find any issue to be solved, so there is no difference between the original structure and the fixed one.   

In [5]:
# # Show protein
# view = nglview.show_structure_file(fixed_pdb)
# view.add_representation(repr_type='ball+stick', selection='all')
# view._remote_call('setSize', target='Widget', args=['','600px'])
# view.camera='orthographic'
# view

<a id="top"></a>
***
## Create protein system topology
**Building GROMACS topology** corresponding to the protein structure.<br>
Force field used in this tutorial is [**amber99sb-ildn**](https://dx.doi.org/10.1002%2Fprot.22711): AMBER **parm99** force field with **corrections on backbone** (sb) and **side-chain torsion potentials** (ildn). Water molecules type used in this tutorial is [**spc/e**](https://pubs.acs.org/doi/abs/10.1021/j100308a038).<br>
Adding **hydrogen atoms** if missing. Automatically identifying **disulfide bridges**. <br>

Generating two output files: 
- **GROMACS structure** (gro file)
- **GROMACS topology** ZIP compressed file containing:
    - *GROMACS topology top file* (top file)
    - *GROMACS position restraint file/s* (itp file/s)
***
**Building Blocks** used:
 - [Pdb2gmx](https://biobb-md.readthedocs.io/en/latest/gromacs.html#module-gromacs.pdb2gmx) from **biobb_gromacs.gromacs.pdb2gmx**
***

In [7]:
# Create system topology
# Import module
from biobb_gromacs.gromacs.pdb2gmx import pdb2gmx

# Create inputs/outputs
output_pdb2gmx_gro = 'Topology/peptide_pdb2gmx.gro'
output_pdb2gmx_top_zip = 'Topology/peptide_pdb2gmx_top.zip'

# Create and launch bb
os.makedirs("Topology")
pdb2gmx(input_pdb_path=peptide_pdb, 
        output_gro_path=output_pdb2gmx_gro, 
        output_top_zip_path=output_pdb2gmx_top_zip)

2026-07-24 11:03:15,154 [MainThread  ] [INFO ]  Module: biobb_gromacs.gromacs.pdb2gmx Version: 5.2.1
2026-07-24 11:03:15,155 [MainThread  ] [INFO ]  Directory successfully created: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_d2cc1194-78e1-4c61-939d-227bf89da184
2026-07-24 11:03:15,155 [MainThread  ] [INFO ]  Copy to stage: peptide.pdb --> sandbox_d2cc1194-78e1-4c61-939d-227bf89da184
2026-07-24 11:03:15,156 [MainThread  ] [INFO ]  Launching command (it may take a while): gmx -nobackup -nocopyright pdb2gmx -f /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_d2cc1194-78e1-4c61-939d-227bf89da184/peptide.pdb -o /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_d2cc1194-78e1-4c61-939d-227bf89da184/peptide_pdb2gmx.gro -p p2g.top -water spce -ff amber99sb-ildn -i posre.itp


2026-07-24 11:03:15,169 [MainThread  ] [INFO ]  Command 'gmx -nobackup -nocopyright pdb2gmx -f /home/lafayette/Escritorio/PeptideCrowding...' finalized with exit code 0
2026-07-24 11:03:15,169 [MainThread  ] [INFO ]  Using the Amber99sb-ildn force field in directory amber99sb-ildn.ff

going to rename amber99sb-ildn.ff/aminoacids.r2b

going to rename amber99sb-ildn.ff/dna.r2b

going to rename amber99sb-ildn.ff/rna.r2b
Reading /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_d2cc1194-78e1-4c61-939d-227bf89da184/peptide.pdb...
Read '4QHU_QSPVLIYDDT_UNCAPPED', 47 atoms

Analyzing pdb file
Splitting chemical chains based on TER records or chain id changing.

There are 1 chains and 0 blocks of water and 7 residues with 47 atoms

  chain  #res #atoms

  1 'L'     7     47  

All occupancies are one

Reading residue database... (Amber99sb-ildn)

Processing chain 1 'L' (47 atoms, 7 residues)

Identified residue ACE43 as a starting terminus.

Identified residue NME49 as a ending

2026-07-24 11:03:15,169 [MainThread  ] [INFO ]                 :-) GROMACS - gmx pdb2gmx, 2025.4-conda_forge (-:

Executable:   /home/lafayette/miniconda3/envs/MD/bin.AVX2_256/gmx
Data prefix:  /home/lafayette/miniconda3/envs/MD
Working dir:  /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide
Command line:
  gmx -nobackup -nocopyright pdb2gmx -f /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_d2cc1194-78e1-4c61-939d-227bf89da184/peptide.pdb -o /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_d2cc1194-78e1-4c61-939d-227bf89da184/peptide_pdb2gmx.gro -p p2g.top -water spce -ff amber99sb-ildn -i posre.itp

Opening force field file /home/lafayette/miniconda3/envs/MD/share/gromacs/top/amber99sb-ildn.ff/aminoacids.r2b
Opening force field file /home/lafayette/miniconda3/envs/MD/share/gromacs/top/amber99sb-ildn.ff/dna.r2b
Opening force field file /home/lafayette/miniconda3/envs/MD/share/gromacs/top/amber99sb-ildn.ff/rna.r2b
All occupancies are one
Openi

2026-07-24 11:03:15,170 [MainThread  ] [INFO ]  Compressing topology to: Topology/peptide_pdb2gmx_top.zip
2026-07-24 11:03:15,170 [MainThread  ] [INFO ]  Ignored file amber99sb-ildn.ff/forcefield.itp
2026-07-24 11:03:15,171 [MainThread  ] [INFO ]  Ignored file amber99sb-ildn.ff/spce.itp
2026-07-24 11:03:15,171 [MainThread  ] [INFO ]  Ignored file amber99sb-ildn.ff/ions.itp
2026-07-24 11:03:15,171 [MainThread  ] [INFO ]  Adding:
2026-07-24 11:03:15,172 [MainThread  ] [INFO ]  ['p2g.top', 'posre.itp']
2026-07-24 11:03:15,172 [MainThread  ] [INFO ]  to: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/Topology/peptide_pdb2gmx_top.zip
2026-07-24 11:03:15,173 [MainThread  ] [INFO ]  Removed: ['posre.itp', 'p2g.top']
2026-07-24 11:03:15,173 [MainThread  ] [INFO ]  Removed: ['/home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_d2cc1194-78e1-4c61-939d-227bf89da184']
2026-07-24 11:03:15,173 [MainThread  ] [INFO ]  


0

### Visualizing 3D structure
Visualizing the generated **GRO structure** using **NGL**. Note that **hydrogen atoms** were added to the structure by the **pdb2gmx GROMACS tool** when generating the **topology**.    

In [8]:
# Show protein
view = nglview.show_structure_file(output_pdb2gmx_gro)
view.add_representation(repr_type='ball+stick', selection='all')
view._remote_call('setSize', target='Widget', args=['','600px'])
view.camera='orthographic'
view

NGLWidget()

<a id="box"></a>
***
## Create solvent box
Define the unit cell for the **protein structure MD system** to fill it with water molecules.<br>
A **cubic box** is used to define the unit cell, with a **distance from the protein to the box edge of 1.0 nm**. The protein is **centered in the box**.  

***
**Building Blocks** used:
 - [Editconf](https://biobb-md.readthedocs.io/en/latest/gromacs.html#module-gromacs.editconf) from **biobb_gromacs.gromacs.editconf** 
***

In [9]:
# Editconf: Create solvent box
# Import module
from biobb_gromacs.gromacs.editconf import editconf

# Create prop dict and inputs/outputs
output_editconf_gro = 'Solvated/crowded_editconf.gro'

prop = {
    'box_type': 'cubic',
    'distance_to_molecule': 1.0
}
os.makedirs("Solvated")
#Create and launch bb
editconf(input_gro_path=output_pdb2gmx_gro, 
         output_gro_path=output_editconf_gro,
         properties=prop)

2026-07-24 11:03:32,140 [MainThread  ] [INFO ]  Module: biobb_gromacs.gromacs.editconf Version: 5.2.1
2026-07-24 11:03:32,140 [MainThread  ] [INFO ]  Directory successfully created: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_c6638930-42e4-4c0e-9982-e24c67be4ac7
2026-07-24 11:03:32,141 [MainThread  ] [INFO ]  Copy to stage: Topology/peptide_pdb2gmx.gro --> sandbox_c6638930-42e4-4c0e-9982-e24c67be4ac7
2026-07-24 11:03:32,141 [MainThread  ] [INFO ]  Distance of the box to molecule:   1.00
2026-07-24 11:03:32,141 [MainThread  ] [INFO ]  Centering molecule in the box.
2026-07-24 11:03:32,141 [MainThread  ] [INFO ]  Box type: cubic
2026-07-24 11:03:32,142 [MainThread  ] [INFO ]  Launching command (it may take a while): gmx -nobackup -nocopyright editconf -f /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_c6638930-42e4-4c0e-9982-e24c67be4ac7/peptide_pdb2gmx.gro -o /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_c6638930-42e4-4c0e-9982

2026-07-24 11:03:32,150 [MainThread  ] [INFO ]                 :-) GROMACS - gmx editconf, 2025.4-conda_forge (-:

Executable:   /home/lafayette/miniconda3/envs/MD/bin.AVX2_256/gmx
Data prefix:  /home/lafayette/miniconda3/envs/MD
Working dir:  /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide
Command line:
  gmx -nobackup -nocopyright editconf -f /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_c6638930-42e4-4c0e-9982-e24c67be4ac7/peptide_pdb2gmx.gro -o /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_c6638930-42e4-4c0e-9982-e24c67be4ac7/crowded_editconf.gro -bt cubic -d 1.0 -c


GROMACS reminds you: "Same sex marriage is not a gay privilege, it's equal rights. Privilege would be something like gay people not paying taxes. Like churches don't." (Ricky Gervais)




2026-07-24 11:03:32,151 [MainThread  ] [INFO ]  Removed: ['/home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_c6638930-42e4-4c0e-9982-e24c67be4ac7']


0

<a id="water"></a>
***
## Fill the box with water molecules
Fill the unit cell for the **protein structure system** with water molecules.<br>
The solvent type used is the default **Simple Point Charge water (SPC)**, a generic equilibrated 3-point solvent model. 

***
**Building Blocks** used:
 - [Solvate](https://biobb-md.readthedocs.io/en/latest/gromacs.html#module-gromacs.solvate) from **biobb_gromacs.gromacs.solvate** 
***

In [10]:
# Solvate: Fill the box with water molecules
from biobb_gromacs.gromacs.solvate import solvate

# Create prop dict and inputs/outputs
output_solvate_gro = 'Solvated/crowded_solvate.gro'
output_solvate_top_zip = 'Solvated/crowded_solvate_top.zip'

# Create and launch bb
solvate(input_solute_gro_path=output_editconf_gro, 
        output_gro_path=output_solvate_gro, 
        input_top_zip_path=output_pdb2gmx_top_zip, 
        output_top_zip_path=output_solvate_top_zip)

2026-07-24 11:03:35,620 [MainThread  ] [INFO ]  Module: biobb_gromacs.gromacs.solvate Version: 5.2.1
2026-07-24 11:03:35,620 [MainThread  ] [INFO ]  Directory successfully created: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_bab74577-2c2c-49e8-8034-b5c827b627f2
2026-07-24 11:03:35,620 [MainThread  ] [INFO ]  Copy to stage: Solvated/crowded_editconf.gro --> sandbox_bab74577-2c2c-49e8-8034-b5c827b627f2
2026-07-24 11:03:35,621 [MainThread  ] [INFO ]  Extracting: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/Topology/peptide_pdb2gmx_top.zip
2026-07-24 11:03:35,621 [MainThread  ] [INFO ]  to:
2026-07-24 11:03:35,621 [MainThread  ] [INFO ]  ['a6b9552f-86b0-40be-b21c-a723a39ac629/p2g.top', 'a6b9552f-86b0-40be-b21c-a723a39ac629/posre.itp']
2026-07-24 11:03:35,621 [MainThread  ] [INFO ]  Unzipping: 
2026-07-24 11:03:35,622 [MainThread  ] [INFO ]  Topology/peptide_pdb2gmx_top.zip
2026-07-24 11:03:35,622 [MainThread  ] [INFO ]  To: 
2026-07-24 11:03:35,622 [MainT

2026-07-24 11:03:35,648 [MainThread  ] [INFO ]                 :-) GROMACS - gmx solvate, 2025.4-conda_forge (-:

Executable:   /home/lafayette/miniconda3/envs/MD/bin.AVX2_256/gmx
Data prefix:  /home/lafayette/miniconda3/envs/MD
Working dir:  /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide
Command line:
  gmx -nobackup -nocopyright solvate -cp /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_bab74577-2c2c-49e8-8034-b5c827b627f2/crowded_editconf.gro -cs spc216.gro -o /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_bab74577-2c2c-49e8-8034-b5c827b627f2/crowded_solvate.gro -p a6b9552f-86b0-40be-b21c-a723a39ac629/p2g.top

Reading solute configuration
Reading solvent configuration

Initialising inter-atomic distances...
Generating solvent configuration
Will generate new solvent configuration of 3x3x3 boxes
Solvent box contains 7119 atoms in 2373 residues
Removed 2010 solvent atoms due to solvent-solvent overlap
Removed 105 solvent atoms due to sol

2026-07-24 11:03:35,649 [MainThread  ] [INFO ]  Compressing topology to: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_bab74577-2c2c-49e8-8034-b5c827b627f2/crowded_solvate_top.zip
2026-07-24 11:03:35,649 [MainThread  ] [INFO ]  Ignored file a6b9552f-86b0-40be-b21c-a723a39ac629/amber99sb-ildn.ff/forcefield.itp
2026-07-24 11:03:35,649 [MainThread  ] [INFO ]  Ignored file a6b9552f-86b0-40be-b21c-a723a39ac629/amber99sb-ildn.ff/spce.itp
2026-07-24 11:03:35,649 [MainThread  ] [INFO ]  Ignored file a6b9552f-86b0-40be-b21c-a723a39ac629/amber99sb-ildn.ff/ions.itp
2026-07-24 11:03:35,650 [MainThread  ] [INFO ]  Adding:
2026-07-24 11:03:35,650 [MainThread  ] [INFO ]  ['a6b9552f-86b0-40be-b21c-a723a39ac629/p2g.top', 'a6b9552f-86b0-40be-b21c-a723a39ac629/posre.itp']
2026-07-24 11:03:35,650 [MainThread  ] [INFO ]  to: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/Solvated/crowded_solvate_top.zip
2026-07-24 11:03:35,650 [MainThread  ] [INFO ]  Removed: ['a6b9552f-86b0-

0

### Visualizing 3D structure
Visualizing the **protein system** with the newly added **solvent box** using **NGL**.<br> Note the **cubic box** filled with **water molecules** surrounding the **protein structure**, which is **centered** right in the middle of the cube.

In [11]:
# Show protein VIEW AT PYMOL
view = nglview.show_structure_file(output_solvate_gro)
view.clear_representations()
view.add_representation(repr_type='cartoon', selection='solute', color='green')
view.add_representation(repr_type='ball+stick', selection='SOL')
view._remote_call('setSize', target='Widget', args=['','600px'])
view.camera='orthographic'
view

NGLWidget()

<a id="ions"></a>
***
## Adding ions
Add ions to neutralize the **protein structure** charge
- [Step 1](#ionsStep1): Creating portable binary run file for ion generation
- [Step 2](#ionsStep2): Adding ions to **neutralize** the system
***
**Building Blocks** used:
 - [Grompp](https://biobb-md.readthedocs.io/en/latest/gromacs.html#module-gromacs.grompp) from **biobb_gromacs.gromacs.grompp** 
 - [Genion](https://biobb-md.readthedocs.io/en/latest/gromacs.html#module-gromacs.genion) from **biobb_gromacs.gromacs.genion** 
***

<a id="ionsStep1"></a>
### Step 1: Creating portable binary run file for ion generation
A simple **energy minimization** molecular dynamics parameters (mdp) properties will be used to generate the portable binary run file for **ion generation**, although **any legitimate combination of parameters** could be used in this step.

In [12]:
# Grompp: Creating portable binary run file for ion generation
from biobb_gromacs.gromacs.grompp import grompp

# Create prop dict and inputs/outputs
output_gppion_tpr = 'Neutralized/crowed_gppion.tpr'
prop = {
    'simulation_type': 'ions',
    'maxwarn': 1
}
os.makedirs("Neutralized")
# Create and launch bb
grompp(input_gro_path=output_solvate_gro, 
       input_top_zip_path=output_solvate_top_zip, 
       output_tpr_path=output_gppion_tpr,  
       properties=prop)

2026-07-24 11:03:55,384 [MainThread  ] [INFO ]  Module: biobb_gromacs.gromacs.grompp Version: 5.2.1
2026-07-24 11:03:55,385 [MainThread  ] [INFO ]  Directory successfully created: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_f6ffed55-6397-4408-872d-cc3ff0571996
2026-07-24 11:03:55,385 [MainThread  ] [INFO ]  Copy to stage: Solvated/crowded_solvate.gro --> sandbox_f6ffed55-6397-4408-872d-cc3ff0571996
2026-07-24 11:03:55,386 [MainThread  ] [INFO ]  Extracting: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/Solvated/crowded_solvate_top.zip
2026-07-24 11:03:55,386 [MainThread  ] [INFO ]  to:
2026-07-24 11:03:55,387 [MainThread  ] [INFO ]  ['/home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_f6ffed55-6397-4408-872d-cc3ff0571996/p2g.top', '/home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_f6ffed55-6397-4408-872d-cc3ff0571996/posre.itp']
2026-07-24 11:03:55,387 [MainThread  ] [INFO ]  Unzipping: 
2026-07-24 11:03:55,387 [MainThrea

2026-07-24 11:03:55,408 [MainThread  ] [INFO ]                  :-) GROMACS - gmx grompp, 2025.4-conda_forge (-:

Executable:   /home/lafayette/miniconda3/envs/MD/bin.AVX2_256/gmx
Data prefix:  /home/lafayette/miniconda3/envs/MD
Working dir:  /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide
Command line:
  gmx -nobackup -nocopyright grompp -f /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_f6ffed55-6397-4408-872d-cc3ff0571996/grompp.mdp -c /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_f6ffed55-6397-4408-872d-cc3ff0571996/crowded_solvate.gro -r /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_f6ffed55-6397-4408-872d-cc3ff0571996/crowded_solvate.gro -p /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_f6ffed55-6397-4408-872d-cc3ff0571996/p2g.top -o /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_f6ffed55-6397-4408-872d-cc3ff0571996/crowed_gppion.tpr -po /home/lafayette/Escritorio/PeptideCrow

2026-07-24 11:03:55,408 [MainThread  ] [INFO ]  Removed: ['/home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_f6ffed55-6397-4408-872d-cc3ff0571996']
2026-07-24 11:03:55,409 [MainThread  ] [INFO ]  


0

<a id="ionsStep2"></a>
### Step 2: Adding ions to neutralize the system
Replace **solvent molecules** with **ions** to **neutralize** the system.

In [13]:
# Genion: Adding ions to neutralize the system
from biobb_gromacs.gromacs.genion import genion

# Create prop dict and inputs/outputs
output_genion_gro = 'Neutralized/crowded_genion.gro'
output_genion_top_zip = 'Neutralized/crowded_genion_top.zip'
prop={
    'neutral':True,
    'concentration':0
}

# Create and launch bb
genion(input_tpr_path=output_gppion_tpr, 
       output_gro_path=output_genion_gro, 
       input_top_zip_path=output_solvate_top_zip, 
       output_top_zip_path=output_genion_top_zip, 
       properties=prop)

2026-07-24 11:03:56,249 [MainThread  ] [INFO ]  Module: biobb_gromacs.gromacs.genion Version: 5.2.1
2026-07-24 11:03:56,250 [MainThread  ] [INFO ]  Directory successfully created: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_1de2e2c0-b29d-4bad-b1d4-077b58926907
2026-07-24 11:03:56,250 [MainThread  ] [INFO ]  Copy to stage: Neutralized/crowed_gppion.tpr --> sandbox_1de2e2c0-b29d-4bad-b1d4-077b58926907
2026-07-24 11:03:56,250 [MainThread  ] [INFO ]  Copy to stage: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/1899380a-add2-465d-9672-f88fa2f37483.stdin --> sandbox_1de2e2c0-b29d-4bad-b1d4-077b58926907
2026-07-24 11:03:56,251 [MainThread  ] [INFO ]  Extracting: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/Solvated/crowded_solvate_top.zip
2026-07-24 11:03:56,251 [MainThread  ] [INFO ]  to:
2026-07-24 11:03:56,251 [MainThread  ] [INFO ]  ['157f01e3-ec80-466a-8060-68ebd188e715/p2g.top', '157f01e3-ec80-466a-8060-68ebd188e715/posre.itp']
2026-07-24 1

2026-07-24 11:03:56,263 [MainThread  ] [INFO ]                  :-) GROMACS - gmx genion, 2025.4-conda_forge (-:

Executable:   /home/lafayette/miniconda3/envs/MD/bin.AVX2_256/gmx
Data prefix:  /home/lafayette/miniconda3/envs/MD
Working dir:  /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide
Command line:
  gmx -nobackup -nocopyright genion -s /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_1de2e2c0-b29d-4bad-b1d4-077b58926907/crowed_gppion.tpr -o /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_1de2e2c0-b29d-4bad-b1d4-077b58926907/crowded_genion.gro -p 157f01e3-ec80-466a-8060-68ebd188e715/p2g.top -neutral -seed 1993

Reading file /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_1de2e2c0-b29d-4bad-b1d4-077b58926907/crowed_gppion.tpr, VERSION 2025.4-conda_forge (single precision)
Reading file /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_1de2e2c0-b29d-4bad-b1d4-077b58926907/crowed_gppion.tpr, VERSION 2025.4-

2026-07-24 11:03:56,264 [MainThread  ] [INFO ]  Compressing topology to: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_1de2e2c0-b29d-4bad-b1d4-077b58926907/crowded_genion_top.zip
2026-07-24 11:03:56,264 [MainThread  ] [INFO ]  Ignored file 157f01e3-ec80-466a-8060-68ebd188e715/amber99sb-ildn.ff/forcefield.itp
2026-07-24 11:03:56,265 [MainThread  ] [INFO ]  Ignored file 157f01e3-ec80-466a-8060-68ebd188e715/amber99sb-ildn.ff/spce.itp
2026-07-24 11:03:56,265 [MainThread  ] [INFO ]  Ignored file 157f01e3-ec80-466a-8060-68ebd188e715/amber99sb-ildn.ff/ions.itp
2026-07-24 11:03:56,266 [MainThread  ] [INFO ]  Adding:
2026-07-24 11:03:56,266 [MainThread  ] [INFO ]  ['157f01e3-ec80-466a-8060-68ebd188e715/p2g.top', '157f01e3-ec80-466a-8060-68ebd188e715/posre.itp']
2026-07-24 11:03:56,266 [MainThread  ] [INFO ]  to: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/Neutralized/crowded_genion_top.zip
2026-07-24 11:03:56,267 [MainThread  ] [INFO ]  Removed: ['157f01e3-ec80

0

### Visualizing 3D structure
Visualizing the **neutralized protein system** with the newly added **ions** using **NGL**

In [14]:
# # Show protein
view = nglview.show_structure_file(output_genion_gro)
view.clear_representations()
view.add_representation(repr_type='cartoon', selection='solute', color='sstruc')
view.add_representation(repr_type='ball+stick', selection='NA')
view.add_representation(repr_type='ball+stick', selection='CL')
view.add_representation(repr_type='ball+stick', selection='SOL')
view._remote_call('setSize', target='Widget', args=['','600px'])
view.camera='orthographic'
view

NGLWidget()

<a id="min"></a>
***
## Energetically minimize the system
Energetically minimize the **protein system** till reaching a desired potential energy.
- [Step 1](#emStep1): Creating portable binary run file for energy minimization
- [Step 2](#emStep2): Energetically minimize the **system** till reaching a force of 500 kJ/mol*nm.
- [Step 3](#emStep3): Checking **energy minimization** results. Plotting energy by time during the **minimization** process.
***
**Building Blocks** used:
 - [Grompp](https://biobb-md.readthedocs.io/en/latest/gromacs.html#module-gromacs.grompp) from **biobb_gromacs.gromacs.grompp** 
 - [Mdrun](https://biobb-md.readthedocs.io/en/latest/gromacs.html#module-gromacs.mdrun) from **biobb_gromacs.gromacs.mdrun** 
 - [GMXEnergy](https://biobb-analysis.readthedocs.io/en/latest/gromacs.html#module-gromacs.gmx_energy) from **biobb_analysis.gromacs.gmx_energy** 
***

<a id="emStep1"></a>
### Step 1: Creating portable binary run file for energy minimization
The **minimization** type of the **molecular dynamics parameters (mdp) property** contains the main default parameters to run an **energy minimization**:

-  integrator  = steep ; Algorithm (steep = steepest descent minimization)
-  emtol       = 1000.0 ; Stop minimization when the maximum force < 1000.0 kJ/mol/nm
-  emstep      = 0.01 ; Minimization step size (nm)
-  nsteps      = 50000 ; Maximum number of (minimization) steps to perform

In this particular example, the method used to run the **energy minimization** is the default **steepest descent**, but the **maximum force** is placed at **500 kJ/mol\*nm^2**, and the **maximum number of steps** to perform (if the maximum force is not reached) to **5,000 steps**. 

In [ ]:
# Grompp: Creating portable binary run file for mdrun
from biobb_gromacs.gromacs.grompp import grompp

# Create prop dict and inputs/outputs
output_gppmin_tpr = 'Emin/peptide_gppmin.tpr'
prop = {
    'mdp':{
        'emtol':'500',
        'nsteps':'100000'
    },
    'simulation_type': 'minimization'
}
os.makedirs("Emin", exist_ok=True)
# Create and launch bb
grompp(input_gro_path=output_genion_gro, 
       input_top_zip_path=output_genion_top_zip, 
       output_tpr_path=output_gppmin_tpr,  
       properties=prop)

FileExistsError: [Errno 17] File exists: 'Emin'

<a id="emStep2"></a>
### Step 2: Running Energy Minimization
Running **energy minimization** using the **tpr file** generated in the previous step. 

In [20]:
# Mdrun: Running minimization
from biobb_gromacs.gromacs.mdrun import mdrun

# Create prop dict and inputs/outputs
output_min_trr = 'Emin/peptide_min.trr'
output_min_gro = 'Emin/peptide_min.gro'
output_min_edr = 'Emin/peptide_min.edr'
output_min_log = 'Emin/peptide_min.log'

# Create and launch bb
mdrun(input_tpr_path=output_gppmin_tpr, 
      output_trr_path=output_min_trr, 
      output_gro_path=output_min_gro, 
      output_edr_path=output_min_edr, 
      output_log_path=output_min_log)

2026-07-24 11:05:01,358 [MainThread  ] [INFO ]  Module: biobb_gromacs.gromacs.mdrun Version: 5.2.1
2026-07-24 11:05:01,358 [MainThread  ] [INFO ]  Directory successfully created: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_63340857-1867-4fb6-b4ee-6983d797fe94
2026-07-24 11:05:01,359 [MainThread  ] [INFO ]  Copy to stage: Emin/peptide_gppmin.tpr --> sandbox_63340857-1867-4fb6-b4ee-6983d797fe94
2026-07-24 11:05:01,359 [MainThread  ] [INFO ]  Launching command (it may take a while): gmx -nobackup -nocopyright mdrun -o /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_63340857-1867-4fb6-b4ee-6983d797fe94/peptide_min.trr -s /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_63340857-1867-4fb6-b4ee-6983d797fe94/peptide_gppmin.tpr -c /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_63340857-1867-4fb6-b4ee-6983d797fe94/peptide_min.gro -e /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_63340857-1867-4fb6-b4e

2026-07-24 11:05:01,807 [MainThread  ] [INFO ]                  :-) GROMACS - gmx mdrun, 2025.4-conda_forge (-:

Executable:   /home/lafayette/miniconda3/envs/MD/bin.AVX2_256/gmx
Data prefix:  /home/lafayette/miniconda3/envs/MD
Working dir:  /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide
Command line:
  gmx -nobackup -nocopyright mdrun -o /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_63340857-1867-4fb6-b4ee-6983d797fe94/peptide_min.trr -s /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_63340857-1867-4fb6-b4ee-6983d797fe94/peptide_gppmin.tpr -c /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_63340857-1867-4fb6-b4ee-6983d797fe94/peptide_min.gro -e /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_63340857-1867-4fb6-b4ee-6983d797fe94/peptide_min.edr -g /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_63340857-1867-4fb6-b4ee-6983d797fe94/peptide_min.log

The current CPU can measure timings m

2026-07-24 11:05:01,808 [MainThread  ] [INFO ]  Removed: ['/home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_63340857-1867-4fb6-b4ee-6983d797fe94']
2026-07-24 11:05:01,808 [MainThread  ] [INFO ]  


0

<a id="emStep3"></a>
### Step 3: Checking Energy Minimization results
Checking **energy minimization** results. Plotting **potential energy** by time during the minimization process. 

In [21]:
# GMXEnergy: Getting system energy by time  
from biobb_analysis.gromacs.gmx_energy import gmx_energy

# Create prop dict and inputs/outputs
output_min_ene_xvg = 'Emin/peptide_min_ene.xvg'
prop = {
    'terms':  ["Potential"]
}

# Create and launch bb
gmx_energy(input_energy_path=output_min_edr, 
          output_xvg_path=output_min_ene_xvg, 
          properties=prop)

2026-07-24 11:05:02,098 [MainThread  ] [INFO ]  Module: biobb_analysis.gromacs.gmx_energy Version: 5.2.1
2026-07-24 11:05:02,098 [MainThread  ] [INFO ]  Directory successfully created: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_cefc6028-f633-4cce-8c10-ec9063db8906
2026-07-24 11:05:02,098 [MainThread  ] [INFO ]  Copy to stage: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/Emin/peptide_min.edr --> sandbox_cefc6028-f633-4cce-8c10-ec9063db8906
2026-07-24 11:05:02,099 [MainThread  ] [INFO ]  Launching command (it may take a while): gmx energy -f /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_cefc6028-f633-4cce-8c10-ec9063db8906/peptide_min.edr -o /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_cefc6028-f633-4cce-8c10-ec9063db8906/peptide_min_ene.xvg -xvg none < /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_cefc6028-f633-4cce-8c10-ec9063db8906/c230a658-59bd-4761-a3ea-e77c3e9b855finstructions.in
2026-0

2026-07-24 11:05:02,109 [MainThread  ] [INFO ]                  :-) GROMACS - gmx energy, 2025.4-conda_forge (-:

Executable:   /home/lafayette/miniconda3/envs/MD/bin.AVX2_256/gmx
Data prefix:  /home/lafayette/miniconda3/envs/MD
Working dir:  /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide
Command line:
  gmx energy -f /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_cefc6028-f633-4cce-8c10-ec9063db8906/peptide_min.edr -o /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_cefc6028-f633-4cce-8c10-ec9063db8906/peptide_min_ene.xvg -xvg none

Opened /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_cefc6028-f633-4cce-8c10-ec9063db8906/peptide_min.edr as single precision energy file

Select the terms you want from the following list by
selecting either (part of) the name or the number or a combination.
End your selection with an empty line or a zero.
-------------------------------------------------------------------
  1  Bond       

2026-07-24 11:05:02,110 [MainThread  ] [INFO ]  Removed: ['/home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_cefc6028-f633-4cce-8c10-ec9063db8906']
2026-07-24 11:05:02,110 [MainThread  ] [INFO ]  


0

In [22]:
import plotly.graph_objs as go

# Read data from file and filter energy values higher than 1000 kJ/mol
with open(output_min_ene_xvg, 'r') as energy_file:
    x, y = zip(*[
        (float(line.split()[0]), float(line.split()[1]))
        for line in energy_file
        if not line.startswith(("#", "@"))
        if float(line.split()[1]) < 1000
    ])

# Create a scatter plot
fig = go.Figure(data=go.Scatter(x=x, y=y, mode='lines'))

# Update layout
fig.update_layout(title="Energy Minimization",
                  xaxis_title="Energy Minimization Step",
                  yaxis_title="Potential Energy kJ/mol",
                  height=600)

# Show the figure
fig.show()

<a id="nvt"></a>
***
## Equilibrate the system (NVT)
Equilibrate the **protein system** in **NVT ensemble** (constant Number of particles, Volume and Temperature). Protein **heavy atoms** will be restrained using position restraining forces: movement is permitted, but only after overcoming a substantial energy penalty. The utility of position restraints is that they allow us to equilibrate our solvent around our protein, without the added variable of structural changes in the protein.

- [Step 1](#eqNVTStep1): Creating portable binary run file for system equilibration
- [Step 2](#eqNVTStep2): Equilibrate the **protein system** with **NVT** ensemble.
- [Step 3](#eqNVTStep3): Checking **NVT Equilibration** results. Plotting **system temperature** by time during the **NVT equilibration** process. 
***
**Building Blocks** used:
- [Grompp](https://biobb-md.readthedocs.io/en/latest/gromacs.html#module-gromacs.grompp) from **biobb_gromacs.gromacs.grompp** 
- [Mdrun](https://biobb-md.readthedocs.io/en/latest/gromacs.html#module-gromacs.mdrun) from **biobb_gromacs.gromacs.mdrun** 
- [GMXEnergy](https://biobb-analysis.readthedocs.io/en/latest/gromacs.html#module-gromacs.gmx_energy) from **biobb_analysis.gromacs.gmx_energy** 
***

<a id="eqNVTStep1"></a>
### Step 1: Creating portable binary run file for system equilibration (NVT)
The **nvt** type of the **molecular dynamics parameters (mdp) property** contains the main default parameters to run an **NVT equilibration** with **protein restraints** (see [GROMACS mdp options](http://manual.gromacs.org/documentation/2018/user-guide/mdp-options.html)):

-  Define                   = -DPOSRES
-  integrator               = md
-  dt                       = 0.002
-  nsteps                   = 5000
-  pcoupl                   = no
-  gen_vel                  = yes
-  gen_temp                 = 300
-  gen_seed                 = -1

In this particular example, the default parameters will be used: **md** integrator algorithm, a **step size** of **2fs**, **5,000 equilibration steps** with the protein **heavy atoms restrained**, and a temperature of **300K**.

*Please note that for the sake of time this tutorial is only running 10ps of NVT equilibration, whereas in the [original example](http://www.mdtutorials.com/gmx/lysozyme/06_equil.html) the simulated time was 100ps.*

In [23]:
# Grompp: Creating portable binary run file for NVT Equilibration
from biobb_gromacs.gromacs.grompp import grompp

# Create prop dict and inputs/outputs
output_gppnvt_tpr = 'Envt/peptide_gppnvt.tpr'
prop = {
    'mdp':{
        'nsteps': 50_000, # 0.002 * 500_000 = 100 ps
        'dt': 0.002,
        # 'Define': '-DPOSRES',
        #'tc_grps': "DNA Water_and_ions" # NOTE: uncomment this line if working with DNA
    },
    'simulation_type': 'nvt'
}
os.makedirs("Envt")
# Create and launch bb
grompp(input_gro_path=output_min_gro, 
       input_top_zip_path=output_genion_top_zip, 
       output_tpr_path=output_gppnvt_tpr,  
       properties=prop)

2026-07-24 11:05:14,312 [MainThread  ] [INFO ]  Module: biobb_gromacs.gromacs.grompp Version: 5.2.1
2026-07-24 11:05:14,313 [MainThread  ] [INFO ]  Directory successfully created: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_08d9cbdc-cd88-46da-9254-4d7676bb17d4
2026-07-24 11:05:14,313 [MainThread  ] [INFO ]  Copy to stage: Emin/peptide_min.gro --> sandbox_08d9cbdc-cd88-46da-9254-4d7676bb17d4
2026-07-24 11:05:14,314 [MainThread  ] [INFO ]  Extracting: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/Neutralized/crowded_genion_top.zip
2026-07-24 11:05:14,315 [MainThread  ] [INFO ]  to:
2026-07-24 11:05:14,315 [MainThread  ] [INFO ]  ['/home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_08d9cbdc-cd88-46da-9254-4d7676bb17d4/p2g.top', '/home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_08d9cbdc-cd88-46da-9254-4d7676bb17d4/posre.itp']
2026-07-24 11:05:14,315 [MainThread  ] [INFO ]  Unzipping: 
2026-07-24 11:05:14,315 [MainThread  ] [

2026-07-24 11:05:14,338 [MainThread  ] [INFO ]                  :-) GROMACS - gmx grompp, 2025.4-conda_forge (-:

Executable:   /home/lafayette/miniconda3/envs/MD/bin.AVX2_256/gmx
Data prefix:  /home/lafayette/miniconda3/envs/MD
Working dir:  /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide
Command line:
  gmx -nobackup -nocopyright grompp -f /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_08d9cbdc-cd88-46da-9254-4d7676bb17d4/grompp.mdp -c /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_08d9cbdc-cd88-46da-9254-4d7676bb17d4/peptide_min.gro -r /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_08d9cbdc-cd88-46da-9254-4d7676bb17d4/peptide_min.gro -p /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_08d9cbdc-cd88-46da-9254-4d7676bb17d4/p2g.top -o /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_08d9cbdc-cd88-46da-9254-4d7676bb17d4/peptide_gppnvt.tpr -po /home/lafayette/Escritorio/PeptideCrowding/MD

2026-07-24 11:05:14,339 [MainThread  ] [INFO ]  Removed: ['/home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_08d9cbdc-cd88-46da-9254-4d7676bb17d4']
2026-07-24 11:05:14,339 [MainThread  ] [INFO ]  


0

<a id="eqNVTStep2"></a>
### Step 2: Running NVT equilibration

In [24]:
# Mdrun: Running Equilibration NVT
from biobb_gromacs.gromacs.mdrun import mdrun

# Create prop dict and inputs/outputs
output_nvt_trr = 'Envt/peptide_nvt.trr'
output_nvt_gro = 'Envt/peptide_nvt.gro'
output_nvt_edr = 'Envt/peptide_nvt.edr'
output_nvt_log = 'Envt/peptide_nvt.log'
output_nvt_cpt = 'Envt/peptide_nvt.cpt'

# Create and launch bb
mdrun(input_tpr_path=output_gppnvt_tpr, 
      output_trr_path=output_nvt_trr, 
      output_gro_path=output_nvt_gro, 
      output_edr_path=output_nvt_edr, 
      output_log_path=output_nvt_log, 
      output_cpt_path=output_nvt_cpt)

2026-07-24 11:05:15,281 [MainThread  ] [INFO ]  Module: biobb_gromacs.gromacs.mdrun Version: 5.2.1
2026-07-24 11:05:15,281 [MainThread  ] [INFO ]  Directory successfully created: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_debb3c49-f7ae-47fd-83cd-36d0182e69a9
2026-07-24 11:05:15,282 [MainThread  ] [INFO ]  Copy to stage: Envt/peptide_gppnvt.tpr --> sandbox_debb3c49-f7ae-47fd-83cd-36d0182e69a9
2026-07-24 11:05:15,282 [MainThread  ] [INFO ]  Launching command (it may take a while): gmx -nobackup -nocopyright mdrun -o /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_debb3c49-f7ae-47fd-83cd-36d0182e69a9/peptide_nvt.trr -s /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_debb3c49-f7ae-47fd-83cd-36d0182e69a9/peptide_gppnvt.tpr -c /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_debb3c49-f7ae-47fd-83cd-36d0182e69a9/peptide_nvt.gro -e /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_debb3c49-f7ae-47fd-83c

2026-07-24 11:05:32,324 [MainThread  ] [INFO ]                  :-) GROMACS - gmx mdrun, 2025.4-conda_forge (-:

Executable:   /home/lafayette/miniconda3/envs/MD/bin.AVX2_256/gmx
Data prefix:  /home/lafayette/miniconda3/envs/MD
Working dir:  /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide
Command line:
  gmx -nobackup -nocopyright mdrun -o /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_debb3c49-f7ae-47fd-83cd-36d0182e69a9/peptide_nvt.trr -s /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_debb3c49-f7ae-47fd-83cd-36d0182e69a9/peptide_gppnvt.tpr -c /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_debb3c49-f7ae-47fd-83cd-36d0182e69a9/peptide_nvt.gro -e /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_debb3c49-f7ae-47fd-83cd-36d0182e69a9/peptide_nvt.edr -g /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_debb3c49-f7ae-47fd-83cd-36d0182e69a9/peptide_nvt.log -cpo /home/lafayette/Escritorio/Peptid

2026-07-24 11:05:32,328 [MainThread  ] [INFO ]  Removed: ['/home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_debb3c49-f7ae-47fd-83cd-36d0182e69a9', 'traj_comp.xtc']
2026-07-24 11:05:32,328 [MainThread  ] [INFO ]  


0

<a id="eqNVTStep3"></a>
### Step 3: Checking NVT Equilibration results
Checking **NVT Equilibration** results. Plotting **system temperature** by time during the NVT equilibration process. 

In [25]:
# GMXEnergy: Getting system temperature by time during NVT Equilibration  
from biobb_analysis.gromacs.gmx_energy import gmx_energy

# Create prop dict and inputs/outputs
output_nvt_temp_xvg = 'Envt/peptide_nvt_temp.xvg'
prop = {
    'terms':  ["Temperature"]
}

# Create and launch bb
gmx_energy(input_energy_path=output_nvt_edr, 
          output_xvg_path=output_nvt_temp_xvg, 
          properties=prop)

2026-07-24 11:06:44,223 [MainThread  ] [INFO ]  Module: biobb_analysis.gromacs.gmx_energy Version: 5.2.1
2026-07-24 11:06:44,224 [MainThread  ] [INFO ]  Directory successfully created: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_be0f3fdd-50ae-4bdd-b58d-62928d9897bc
2026-07-24 11:06:44,224 [MainThread  ] [INFO ]  Copy to stage: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/Envt/peptide_nvt.edr --> sandbox_be0f3fdd-50ae-4bdd-b58d-62928d9897bc
2026-07-24 11:06:44,225 [MainThread  ] [INFO ]  Launching command (it may take a while): gmx energy -f /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_be0f3fdd-50ae-4bdd-b58d-62928d9897bc/peptide_nvt.edr -o /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_be0f3fdd-50ae-4bdd-b58d-62928d9897bc/peptide_nvt_temp.xvg -xvg none < /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_be0f3fdd-50ae-4bdd-b58d-62928d9897bc/13f34140-18ba-4fec-8ef4-e71d2beaec71instructions.in
2026-

2026-07-24 11:06:44,235 [MainThread  ] [INFO ]                  :-) GROMACS - gmx energy, 2025.4-conda_forge (-:

Executable:   /home/lafayette/miniconda3/envs/MD/bin.AVX2_256/gmx
Data prefix:  /home/lafayette/miniconda3/envs/MD
Working dir:  /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide
Command line:
  gmx energy -f /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_be0f3fdd-50ae-4bdd-b58d-62928d9897bc/peptide_nvt.edr -o /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_be0f3fdd-50ae-4bdd-b58d-62928d9897bc/peptide_nvt_temp.xvg -xvg none

Opened /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_be0f3fdd-50ae-4bdd-b58d-62928d9897bc/peptide_nvt.edr as single precision energy file

Select the terms you want from the following list by
selecting either (part of) the name or the number or a combination.
End your selection with an empty line or a zero.
-------------------------------------------------------------------
  1  Bond      

2026-07-24 11:06:44,236 [MainThread  ] [INFO ]  Removed: ['/home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_be0f3fdd-50ae-4bdd-b58d-62928d9897bc']
2026-07-24 11:06:44,236 [MainThread  ] [INFO ]  


0

In [26]:
import plotly.graph_objs as go

# Read temperature data from file
with open(output_nvt_temp_xvg, 'r') as temperature_file:
    x, y = zip(*[
        (float(line.split()[0]), float(line.split()[1]))
        for line in temperature_file
        if not line.startswith(("#", "@"))
    ])

# Create a scatter plot
fig = go.Figure(data=go.Scatter(x=x, y=y, mode='lines+markers'))

# Update layout
fig.update_layout(title="Temperature during NVT Equilibration",
                  xaxis_title="Time (ps)",
                  yaxis_title="Temperature (K)",
                  height=600)

# Show the figure
fig.show()

<a id="npt"></a>
***
## Equilibrate the system (NPT)
Equilibrate the **protein system** in **NPT** ensemble (constant Number of particles, Pressure and Temperature).
- [Step 1](#eqNPTStep1): Creating portable binary run file for system equilibration
- [Step 2](#eqNPTStep2): Equilibrate the **protein system** with **NPT** ensemble.
- [Step 3](#eqNPTStep3): Checking **NPT Equilibration** results. Plotting **system pressure and density** by time during the **NPT equilibration** process.
***
**Building Blocks** used:
 - [Grompp](https://biobb-md.readthedocs.io/en/latest/gromacs.html#module-gromacs.grompp) from **biobb_gromacs.gromacs.grompp** 
 - [Mdrun](https://biobb-md.readthedocs.io/en/latest/gromacs.html#module-gromacs.mdrun) from **biobb_gromacs.gromacs.mdrun** 
 - [GMXEnergy](https://biobb-analysis.readthedocs.io/en/latest/gromacs.html#module-gromacs.gmx_energy) from **biobb_analysis.gromacs.gmx_energy** 
***

<a id="eqNPTStep1"></a>
### Step 1: Creating portable binary run file for system equilibration (NPT)

The **npt** type of the **molecular dynamics parameters (mdp) property** contains the main default parameters to run an **NPT equilibration** with **protein restraints** (see [GROMACS mdp options](http://manual.gromacs.org/documentation/2018/user-guide/mdp-options.html)):

-  Define                   = -DPOSRES
-  integrator               = md
-  dt                       = 0.002
-  nsteps                   = 5000
-  pcoupl = Parrinello-Rahman
-  pcoupltype = isotropic
-  tau_p = 1.0
-  ref_p = 1.0
-  compressibility = 4.5e-5
-  refcoord_scaling = com
-  gen_vel = no

In this particular example, the default parameters will be used: **md** integrator algorithm, a **time step** of **2fs**, **5,000 equilibration steps** with the protein **heavy atoms restrained**, and a Parrinello-Rahman **pressure coupling** algorithm.

*Please note that for the sake of time this tutorial is only running 10ps of NPT equilibration, whereas in the [original example](http://www.mdtutorials.com/gmx/lysozyme/07_equil2.html) the simulated time was 100ps.*

In [ ]:
# Grompp: Creating portable binary run file for NPT System Equilibration
from biobb_gromacs.gromacs.grompp import grompp

# Create prop dict and inputs/outputs
output_gppnpt_tpr = 'Enpt/peptide_gppnpt.tpr'
prop = {
    'mdp':{
        'nsteps':'10000',
        #'tc_grps': "DNA Water_and_ions" # NOTE: uncomment this line if working with DNA
    },
    'simulation_type': 'npt'
}
os.makedirs("Enpt", exist_ok=True)
# Create and launch bb
grompp(input_gro_path=output_nvt_gro, 
       input_top_zip_path=output_genion_top_zip, 
       output_tpr_path=output_gppnpt_tpr, 
       input_cpt_path=output_nvt_cpt,  
       properties=prop)

2026-07-24 11:07:00,632 [MainThread  ] [INFO ]  Module: biobb_gromacs.gromacs.grompp Version: 5.2.1
2026-07-24 11:07:00,633 [MainThread  ] [INFO ]  Directory successfully created: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_3b3ba12f-7945-4872-ad46-70f0aeaf99dc
2026-07-24 11:07:00,633 [MainThread  ] [INFO ]  Copy to stage: Envt/peptide_nvt.gro --> sandbox_3b3ba12f-7945-4872-ad46-70f0aeaf99dc
2026-07-24 11:07:00,633 [MainThread  ] [INFO ]  Copy to stage: Envt/peptide_nvt.cpt --> sandbox_3b3ba12f-7945-4872-ad46-70f0aeaf99dc
2026-07-24 11:07:00,634 [MainThread  ] [INFO ]  Extracting: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/Neutralized/crowded_genion_top.zip
2026-07-24 11:07:00,634 [MainThread  ] [INFO ]  to:
2026-07-24 11:07:00,634 [MainThread  ] [INFO ]  ['/home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_3b3ba12f-7945-4872-ad46-70f0aeaf99dc/p2g.top', '/home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_3b3ba12f-7945-48

2026-07-24 11:07:00,667 [MainThread  ] [INFO ]                  :-) GROMACS - gmx grompp, 2025.4-conda_forge (-:

Executable:   /home/lafayette/miniconda3/envs/MD/bin.AVX2_256/gmx
Data prefix:  /home/lafayette/miniconda3/envs/MD
Working dir:  /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide
Command line:
  gmx -nobackup -nocopyright grompp -f /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_3b3ba12f-7945-4872-ad46-70f0aeaf99dc/grompp.mdp -c /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_3b3ba12f-7945-4872-ad46-70f0aeaf99dc/peptide_nvt.gro -r /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_3b3ba12f-7945-4872-ad46-70f0aeaf99dc/peptide_nvt.gro -p /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_3b3ba12f-7945-4872-ad46-70f0aeaf99dc/p2g.top -o /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_3b3ba12f-7945-4872-ad46-70f0aeaf99dc/peptide_gppnpt.tpr -po /home/lafayette/Escritorio/PeptideCrowding/MD

2026-07-24 11:07:00,667 [MainThread  ] [INFO ]  Removed: ['/home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_3b3ba12f-7945-4872-ad46-70f0aeaf99dc']
2026-07-24 11:07:00,667 [MainThread  ] [INFO ]  


0

<a id="eqNPTStep2"></a>
### Step 2: Running NPT equilibration

In [28]:
# Mdrun: Running NPT System Equilibration
from biobb_gromacs.gromacs.mdrun import mdrun

# Create prop dict and inputs/outputs
output_npt_trr = 'Enpt/peptide_npt.trr'
output_npt_gro = 'Enpt/peptide_npt.gro'
output_npt_edr = 'Enpt/peptide_npt.edr'
output_npt_log = 'Enpt/peptide_npt.log'
output_npt_cpt = 'Enpt/peptide_npt.cpt'

# Create and launch bb
mdrun(input_tpr_path=output_gppnpt_tpr, 
      output_trr_path=output_npt_trr, 
      output_gro_path=output_npt_gro, 
      output_edr_path=output_npt_edr, 
      output_log_path=output_npt_log, 
      output_cpt_path=output_npt_cpt)

2026-07-24 11:07:02,151 [MainThread  ] [INFO ]  Module: biobb_gromacs.gromacs.mdrun Version: 5.2.1
2026-07-24 11:07:02,151 [MainThread  ] [INFO ]  Directory successfully created: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_0e434519-edb0-49ef-bbf7-64fa4d22c525
2026-07-24 11:07:02,152 [MainThread  ] [INFO ]  Copy to stage: Enpt/peptide_gppnpt.tpr --> sandbox_0e434519-edb0-49ef-bbf7-64fa4d22c525
2026-07-24 11:07:02,152 [MainThread  ] [INFO ]  Launching command (it may take a while): gmx -nobackup -nocopyright mdrun -o /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_0e434519-edb0-49ef-bbf7-64fa4d22c525/peptide_npt.trr -s /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_0e434519-edb0-49ef-bbf7-64fa4d22c525/peptide_gppnpt.tpr -c /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_0e434519-edb0-49ef-bbf7-64fa4d22c525/peptide_npt.gro -e /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_0e434519-edb0-49ef-bbf

2026-07-24 11:07:05,261 [MainThread  ] [INFO ]                  :-) GROMACS - gmx mdrun, 2025.4-conda_forge (-:

Executable:   /home/lafayette/miniconda3/envs/MD/bin.AVX2_256/gmx
Data prefix:  /home/lafayette/miniconda3/envs/MD
Working dir:  /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide
Command line:
  gmx -nobackup -nocopyright mdrun -o /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_0e434519-edb0-49ef-bbf7-64fa4d22c525/peptide_npt.trr -s /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_0e434519-edb0-49ef-bbf7-64fa4d22c525/peptide_gppnpt.tpr -c /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_0e434519-edb0-49ef-bbf7-64fa4d22c525/peptide_npt.gro -e /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_0e434519-edb0-49ef-bbf7-64fa4d22c525/peptide_npt.edr -g /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_0e434519-edb0-49ef-bbf7-64fa4d22c525/peptide_npt.log -cpo /home/lafayette/Escritorio/Peptid

2026-07-24 11:07:05,262 [MainThread  ] [INFO ]  Removed: ['/home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_0e434519-edb0-49ef-bbf7-64fa4d22c525', 'traj_comp.xtc']
2026-07-24 11:07:05,263 [MainThread  ] [INFO ]  


0

<a id="eqNPTStep3"></a>
### Step 3: Checking NPT Equilibration results
Checking **NPT Equilibration** results. Plotting **system pressure and density** by time during the **NPT equilibration** process. 

In [29]:
# GMXEnergy: Getting system pressure and density by time during NPT Equilibration  
from biobb_analysis.gromacs.gmx_energy import gmx_energy

# Create prop dict and inputs/outputs
output_npt_pd_xvg = 'Enpt/peptide_npt_PD.xvg'
prop = {
    'terms':  ["Pressure","Density"]
}

# Create and launch bb
gmx_energy(input_energy_path=output_npt_edr, 
          output_xvg_path=output_npt_pd_xvg, 
          properties=prop)

2026-07-24 11:07:09,900 [MainThread  ] [INFO ]  Module: biobb_analysis.gromacs.gmx_energy Version: 5.2.1
2026-07-24 11:07:09,900 [MainThread  ] [INFO ]  Directory successfully created: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_d4df180b-534e-42a6-8fff-7a3d111119e5
2026-07-24 11:07:09,901 [MainThread  ] [INFO ]  Copy to stage: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/Enpt/peptide_npt.edr --> sandbox_d4df180b-534e-42a6-8fff-7a3d111119e5
2026-07-24 11:07:09,901 [MainThread  ] [INFO ]  Launching command (it may take a while): gmx energy -f /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_d4df180b-534e-42a6-8fff-7a3d111119e5/peptide_npt.edr -o /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_d4df180b-534e-42a6-8fff-7a3d111119e5/peptide_npt_PD.xvg -xvg none < /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_d4df180b-534e-42a6-8fff-7a3d111119e5/ae0db2cf-04de-4a5d-8bc2-cc7dec730067instructions.in
2026-07

2026-07-24 11:07:09,911 [MainThread  ] [INFO ]                  :-) GROMACS - gmx energy, 2025.4-conda_forge (-:

Executable:   /home/lafayette/miniconda3/envs/MD/bin.AVX2_256/gmx
Data prefix:  /home/lafayette/miniconda3/envs/MD
Working dir:  /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide
Command line:
  gmx energy -f /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_d4df180b-534e-42a6-8fff-7a3d111119e5/peptide_npt.edr -o /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_d4df180b-534e-42a6-8fff-7a3d111119e5/peptide_npt_PD.xvg -xvg none

Opened /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_d4df180b-534e-42a6-8fff-7a3d111119e5/peptide_npt.edr as single precision energy file

Select the terms you want from the following list by
selecting either (part of) the name or the number or a combination.
End your selection with an empty line or a zero.
-------------------------------------------------------------------
  1  Bond        

2026-07-24 11:07:09,912 [MainThread  ] [INFO ]  Removed: ['/home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_d4df180b-534e-42a6-8fff-7a3d111119e5']
2026-07-24 11:07:09,912 [MainThread  ] [INFO ]  


0

In [30]:
import plotly.graph_objs as go
from plotly.subplots import make_subplots

# Read pressure and density data from file 
with open(output_npt_pd_xvg,'r') as pd_file:
    x, y, z = zip(*[
        (float(line.split()[0]), float(line.split()[1]), float(line.split()[2]))
        for line in pd_file
        if not line.startswith(("#", "@"))
    ])

trace1 = go.Scatter(
    x=x,y=y
)
trace2 = go.Scatter(
    x=x,y=z
)

fig = make_subplots(rows=1, cols=2, print_grid=False)
fig.append_trace(trace1, 1, 1)
fig.append_trace(trace2, 1, 2)

fig.update_layout(
    height=500,
    title='Pressure and Density during NPT Equilibration',
    showlegend=False,
    xaxis1_title='Time (ps)',
    yaxis1_title='Pressure (bar)',
    xaxis2_title='Time (ps)',
    yaxis2_title='Density (Kg*m^-3)'
)

# Show the figure
fig.show()

<a id="free"></a>
***
## Free Molecular Dynamics Simulation
Upon completion of the **two equilibration phases (NVT and NPT)**, the system is now well-equilibrated at the desired temperature and pressure. The **position restraints** can now be released. The last step of the **protein** MD setup is a short, **free MD simulation**, to ensure the robustness of the system. 
- [Step 1](#mdStep1): Creating portable binary run file to run a **free MD simulation**.
- [Step 2](#mdStep2): Run short MD simulation of the **protein system**.
- [Step 3](#mdStep3): Checking results for the final step of the setup process, the **free MD run**. Plotting **Root Mean Square deviation (RMSd)** and **Radius of Gyration (Rgyr)** by time during the **free MD run** step. 
***
**Building Blocks** used:
 - [Grompp](https://biobb-md.readthedocs.io/en/latest/gromacs.html#module-gromacs.grompp) from **biobb_gromacs.gromacs.grompp** 
 - [Mdrun](https://biobb-md.readthedocs.io/en/latest/gromacs.html#module-gromacs.mdrun) from **biobb_gromacs.gromacs.mdrun** 
 - [GMXRms](https://biobb-analysis.readthedocs.io/en/latest/gromacs.html#module-gromacs.gmx_rms) from **biobb_analysis.gromacs.gmx_rms** 
 - [GMXRgyr](https://biobb-analysis.readthedocs.io/en/latest/gromacs.html#module-gromacs.gmx_rgyr) from **biobb_analysis.gromacs.gmx_rgyr** 
***

<a id="mdStep1"></a>
### Step 1: Creating portable binary run file to run a free MD simulation

The **free** type of the **molecular dynamics parameters (mdp) property** contains the main default parameters to run an **free MD simulation** (see [GROMACS mdp options](http://manual.gromacs.org/documentation/2018/user-guide/mdp-options.html)):

-  integrator               = md
-  dt                       = 0.002 (ps)
-  nsteps                   = 50000

In this particular example, the default parameters will be used: **md** integrator algorithm, a **time step** of **2fs**, and a total of **50,000 md steps** (100ps).

*Please note that for the sake of time this tutorial is only running 100ps of free MD, whereas in the [original example](http://www.mdtutorials.com/gmx/lysozyme/08_MD.html) the simulated time was 1ns (1000ps).*

In [38]:
# Grompp: Creating portable binary run file for mdrun
from biobb_gromacs.gromacs.grompp import grompp

# Create prop dict and inputs/outputs
output_gppmd_tpr = 'MD_sym/peptide_gppmd.tpr'
prop = {
    'mdp':{
        'nsteps':'5000000',
        #'tc_grps': "DNA Water_and_ions" # NOTE: uncomment this line if working with DNA
    },
    'simulation_type': 'free'
}
os.makedirs("MD_sym", exist_ok=True)
# Create and launch bb
grompp(input_gro_path=output_npt_gro, 
       input_top_zip_path=output_genion_top_zip, 
       output_tpr_path=output_gppmd_tpr, 
       input_cpt_path=output_npt_cpt, 
       properties=prop)

2026-07-24 11:10:27,871 [MainThread  ] [INFO ]  Module: biobb_gromacs.gromacs.grompp Version: 5.2.1
2026-07-24 11:10:27,872 [MainThread  ] [INFO ]  Directory successfully created: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_7820cad3-3cb6-4f23-9568-49b2d286a5dc
2026-07-24 11:10:27,872 [MainThread  ] [INFO ]  Copy to stage: Enpt/peptide_npt.gro --> sandbox_7820cad3-3cb6-4f23-9568-49b2d286a5dc
2026-07-24 11:10:27,873 [MainThread  ] [INFO ]  Copy to stage: Enpt/peptide_npt.cpt --> sandbox_7820cad3-3cb6-4f23-9568-49b2d286a5dc
2026-07-24 11:10:27,873 [MainThread  ] [INFO ]  Extracting: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/Neutralized/crowded_genion_top.zip
2026-07-24 11:10:27,873 [MainThread  ] [INFO ]  to:
2026-07-24 11:10:27,873 [MainThread  ] [INFO ]  ['/home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_7820cad3-3cb6-4f23-9568-49b2d286a5dc/p2g.top', '/home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_7820cad3-3cb6-4f

2026-07-24 11:10:27,897 [MainThread  ] [INFO ]                  :-) GROMACS - gmx grompp, 2025.4-conda_forge (-:

Executable:   /home/lafayette/miniconda3/envs/MD/bin.AVX2_256/gmx
Data prefix:  /home/lafayette/miniconda3/envs/MD
Working dir:  /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide
Command line:
  gmx -nobackup -nocopyright grompp -f /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_7820cad3-3cb6-4f23-9568-49b2d286a5dc/grompp.mdp -c /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_7820cad3-3cb6-4f23-9568-49b2d286a5dc/peptide_npt.gro -r /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_7820cad3-3cb6-4f23-9568-49b2d286a5dc/peptide_npt.gro -p /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_7820cad3-3cb6-4f23-9568-49b2d286a5dc/p2g.top -o /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_7820cad3-3cb6-4f23-9568-49b2d286a5dc/peptide_gppmd.tpr -po /home/lafayette/Escritorio/PeptideCrowding/MD_

2026-07-24 11:10:27,898 [MainThread  ] [INFO ]  Removed: ['/home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_7820cad3-3cb6-4f23-9568-49b2d286a5dc']
2026-07-24 11:10:27,898 [MainThread  ] [INFO ]  


0

<a id="mdStep2"></a>
### Step 2: Running short free MD simulation

In [39]:
# Mdrun: Running free dynamics
from biobb_gromacs.gromacs.mdrun import mdrun

# Create prop dict and inputs/outputs
output_md_trr = 'MD_sym/peptide_md.trr'
output_md_gro = 'MD_sym/peptide_md.gro'
output_md_edr = 'MD_sym/peptide_md.edr'
output_md_log = 'MD_sym/peptide_md.log'
output_md_cpt = 'MD_sym/peptide_md.cpt'

# Create and launch bb
mdrun(input_tpr_path=output_gppmd_tpr, 
      output_trr_path=output_md_trr, 
      output_gro_path=output_md_gro, 
      output_edr_path=output_md_edr, 
      output_log_path=output_md_log, 
      output_cpt_path=output_md_cpt)

2026-07-24 11:10:38,189 [MainThread  ] [INFO ]  Module: biobb_gromacs.gromacs.mdrun Version: 5.2.1
2026-07-24 11:10:38,190 [MainThread  ] [INFO ]  Directory successfully created: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_578be560-79d7-44b3-bde4-f8e0882e431c
2026-07-24 11:10:38,190 [MainThread  ] [INFO ]  Copy to stage: MD_sym/peptide_gppmd.tpr --> sandbox_578be560-79d7-44b3-bde4-f8e0882e431c
2026-07-24 11:10:38,191 [MainThread  ] [INFO ]  Launching command (it may take a while): gmx -nobackup -nocopyright mdrun -o /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_578be560-79d7-44b3-bde4-f8e0882e431c/peptide_md.trr -s /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_578be560-79d7-44b3-bde4-f8e0882e431c/peptide_gppmd.tpr -c /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_578be560-79d7-44b3-bde4-f8e0882e431c/peptide_md.gro -e /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_578be560-79d7-44b3-bde4-

2026-07-24 11:40:27,008 [MainThread  ] [INFO ]                  :-) GROMACS - gmx mdrun, 2025.4-conda_forge (-:

Executable:   /home/lafayette/miniconda3/envs/MD/bin.AVX2_256/gmx
Data prefix:  /home/lafayette/miniconda3/envs/MD
Working dir:  /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide
Command line:
  gmx -nobackup -nocopyright mdrun -o /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_578be560-79d7-44b3-bde4-f8e0882e431c/peptide_md.trr -s /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_578be560-79d7-44b3-bde4-f8e0882e431c/peptide_gppmd.tpr -c /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_578be560-79d7-44b3-bde4-f8e0882e431c/peptide_md.gro -e /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_578be560-79d7-44b3-bde4-f8e0882e431c/peptide_md.edr -g /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_578be560-79d7-44b3-bde4-f8e0882e431c/peptide_md.log -cpo /home/lafayette/Escritorio/PeptideCrow

2026-07-24 11:40:27,064 [MainThread  ] [INFO ]  Removed: ['/home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_578be560-79d7-44b3-bde4-f8e0882e431c', 'traj_comp.xtc']
2026-07-24 11:40:27,064 [MainThread  ] [INFO ]  


0

<a id="mdStep3"></a>
### Step 3: Checking free MD simulation results
Checking results for the final step of the setup process, the **free MD run**. Plotting **Root Mean Square deviation (RMSd)** and **Radius of Gyration (Rgyr)** by time during the **free MD run** step. **RMSd** against the **experimental structure** (input structure of the pipeline) and against the **minimized and equilibrated structure** (output structure of the NPT equilibration step).

In [40]:
# GMXRms: Computing Root Mean Square deviation to analyse structural stability 
#         RMSd against minimized and equilibrated snapshot (backbone atoms)   

from biobb_analysis.gromacs.gmx_rms import gmx_rms

# Create prop dict and inputs/outputs
output_rms_first = 'MD_sym/peptide_rms_first.xvg'
prop = {
    'selection':  'Backbone',
    # 'selection': 'non-Water'
}

# Create and launch bb
gmx_rms(input_structure_path=output_gppmd_tpr,
         input_traj_path=output_md_trr,
         output_xvg_path=output_rms_first, 
          properties=prop)

2026-07-24 11:42:51,233 [MainThread  ] [INFO ]  Module: biobb_analysis.gromacs.gmx_rms Version: 5.2.1
2026-07-24 11:42:51,234 [MainThread  ] [INFO ]  Directory successfully created: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_375e5a30-61c0-4c46-8995-496fb205f399
2026-07-24 11:42:51,234 [MainThread  ] [INFO ]  Copy to stage: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/MD_sym/peptide_gppmd.tpr --> sandbox_375e5a30-61c0-4c46-8995-496fb205f399
2026-07-24 11:42:51,235 [MainThread  ] [INFO ]  Copy to stage: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/MD_sym/peptide_md.trr --> sandbox_375e5a30-61c0-4c46-8995-496fb205f399
2026-07-24 11:42:51,271 [MainThread  ] [INFO ]  Copy to stage: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/3ca79c29-1f66-4b9a-9104-4fa5df48a397.stdin --> sandbox_375e5a30-61c0-4c46-8995-496fb205f399
2026-07-24 11:42:51,271 [MainThread  ] [INFO ]  Launching command (it may take a while): gmx rms -s /home/lafayette

2026-07-24 11:42:51,602 [MainThread  ] [INFO ]                   :-) GROMACS - gmx rms, 2025.4-conda_forge (-:

Executable:   /home/lafayette/miniconda3/envs/MD/bin.AVX2_256/gmx
Data prefix:  /home/lafayette/miniconda3/envs/MD
Working dir:  /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide
Command line:
  gmx rms -s /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_375e5a30-61c0-4c46-8995-496fb205f399/peptide_gppmd.tpr -f /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_375e5a30-61c0-4c46-8995-496fb205f399/peptide_md.trr -o /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_375e5a30-61c0-4c46-8995-496fb205f399/peptide_rms_first.xvg -xvg none

Reading file /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_375e5a30-61c0-4c46-8995-496fb205f399/peptide_gppmd.tpr, VERSION 2025.4-conda_forge (single precision)
Reading file /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_375e5a30-61c0-4c46-8995-496fb205f

2026-07-24 11:42:51,606 [MainThread  ] [INFO ]  Removed: ['/home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_375e5a30-61c0-4c46-8995-496fb205f399', '/home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/3ca79c29-1f66-4b9a-9104-4fa5df48a397.stdin']
2026-07-24 11:42:51,606 [MainThread  ] [INFO ]  


0

In [41]:
# GMXRms: Computing Root Mean Square deviation to analyse structural stability 
#         RMSd against experimental structure (backbone atoms)   

from biobb_analysis.gromacs.gmx_rms import gmx_rms

# Create prop dict and inputs/outputs
output_rms_exp = 'MD_sym/peptide_rms_exp.xvg'
prop = {
    'selection':  'Backbone',
    #'selection': 'non-Water'
}

# Create and launch bb
gmx_rms(input_structure_path=output_gppmin_tpr,
         input_traj_path=output_md_trr,
         output_xvg_path=output_rms_exp, 
          properties=prop)

2026-07-24 11:42:52,288 [MainThread  ] [INFO ]  Module: biobb_analysis.gromacs.gmx_rms Version: 5.2.1
2026-07-24 11:42:52,288 [MainThread  ] [INFO ]  Directory successfully created: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_eb4ef613-4792-452e-9e99-463f301c17c6
2026-07-24 11:42:52,289 [MainThread  ] [INFO ]  Copy to stage: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/Emin/peptide_gppmin.tpr --> sandbox_eb4ef613-4792-452e-9e99-463f301c17c6
2026-07-24 11:42:52,289 [MainThread  ] [INFO ]  Copy to stage: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/MD_sym/peptide_md.trr --> sandbox_eb4ef613-4792-452e-9e99-463f301c17c6
2026-07-24 11:42:52,311 [MainThread  ] [INFO ]  Copy to stage: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/93da00e4-7ff2-46a3-96c0-98590734fd88.stdin --> sandbox_eb4ef613-4792-452e-9e99-463f301c17c6
2026-07-24 11:42:52,311 [MainThread  ] [INFO ]  Launching command (it may take a while): gmx rms -s /home/lafayette/

2026-07-24 11:42:52,635 [MainThread  ] [INFO ]                   :-) GROMACS - gmx rms, 2025.4-conda_forge (-:

Executable:   /home/lafayette/miniconda3/envs/MD/bin.AVX2_256/gmx
Data prefix:  /home/lafayette/miniconda3/envs/MD
Working dir:  /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide
Command line:
  gmx rms -s /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_eb4ef613-4792-452e-9e99-463f301c17c6/peptide_gppmin.tpr -f /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_eb4ef613-4792-452e-9e99-463f301c17c6/peptide_md.trr -o /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_eb4ef613-4792-452e-9e99-463f301c17c6/peptide_rms_exp.xvg -xvg none

Reading file /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_eb4ef613-4792-452e-9e99-463f301c17c6/peptide_gppmin.tpr, VERSION 2025.4-conda_forge (single precision)
Reading file /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_eb4ef613-4792-452e-9e99-463f301c1

2026-07-24 11:42:52,639 [MainThread  ] [INFO ]  Removed: ['/home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_eb4ef613-4792-452e-9e99-463f301c17c6', '/home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/93da00e4-7ff2-46a3-96c0-98590734fd88.stdin']
2026-07-24 11:42:52,639 [MainThread  ] [INFO ]  


0

In [42]:
import plotly.graph_objs as go

# Read RMS vs first snapshot data from file 
with open(output_rms_first,'r') as rms_first_file:
    x, y = zip(*[
        (float(line.split()[0]), float(line.split()[1]))
        for line in rms_first_file
        if not line.startswith(("#", "@"))
    ])

# Read RMS vs experimental structure data from file 
with open(output_rms_exp,'r') as rms_exp_file:
    x2, y2 = zip(*[
        (float(line.split()[0]), float(line.split()[1]))
        for line in rms_exp_file
        if not line.startswith(("#", "@"))
    ])

fig = make_subplots()
fig.add_trace(go.Scatter(x=x, y=y, mode="lines+markers", name="RMSd vs first"))
fig.add_trace(go.Scatter(x=x, y=y2, mode="lines+markers", name="RMSd vs exp"))

# Set layout including height
fig.update_layout(
    title="RMSd during free MD Simulation",
    xaxis=dict(title="Time (ps)"),
    yaxis=dict(title="RMSd (nm)"),
    height=600
)

# Show the figure
fig.show()

In [43]:
# GMXRgyr: Computing Radius of Gyration to measure the protein compactness during the free MD simulation 

from biobb_analysis.gromacs.gmx_rgyr import gmx_rgyr

# Create prop dict and inputs/outputs
output_rgyr = 'MD_sym/peptide_rgyr.xvg'
prop = {
    'selection':  'Backbone'
}

# Create and launch bb
gmx_rgyr(input_structure_path=output_gppmin_tpr,
         input_traj_path=output_md_trr,
         output_xvg_path=output_rgyr, 
          properties=prop)

2026-07-24 11:42:53,778 [MainThread  ] [INFO ]  Module: biobb_analysis.gromacs.gmx_rgyr Version: 5.2.1
2026-07-24 11:42:53,779 [MainThread  ] [INFO ]  Directory successfully created: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_7d068895-4af2-4660-adfe-699247ead240
2026-07-24 11:42:53,779 [MainThread  ] [INFO ]  Copy to stage: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/Emin/peptide_gppmin.tpr --> sandbox_7d068895-4af2-4660-adfe-699247ead240
2026-07-24 11:42:53,779 [MainThread  ] [INFO ]  Copy to stage: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/MD_sym/peptide_md.trr --> sandbox_7d068895-4af2-4660-adfe-699247ead240
2026-07-24 11:42:53,821 [MainThread  ] [INFO ]  Copy to stage: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/468e3b7e-83ff-4f24-8a90-39c843898340.stdin --> sandbox_7d068895-4af2-4660-adfe-699247ead240
2026-07-24 11:42:53,822 [MainThread  ] [INFO ]  Launching command (it may take a while): gmx gyrate -s /home/lafaye

2026-07-24 11:42:54,143 [MainThread  ] [INFO ]                  :-) GROMACS - gmx gyrate, 2025.4-conda_forge (-:

Executable:   /home/lafayette/miniconda3/envs/MD/bin.AVX2_256/gmx
Data prefix:  /home/lafayette/miniconda3/envs/MD
Working dir:  /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide
Command line:
  gmx gyrate -s /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_7d068895-4af2-4660-adfe-699247ead240/peptide_gppmin.tpr -f /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_7d068895-4af2-4660-adfe-699247ead240/peptide_md.trr -o /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_7d068895-4af2-4660-adfe-699247ead240/peptide_rgyr.xvg -xvg none

Reading file /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_7d068895-4af2-4660-adfe-699247ead240/peptide_gppmin.tpr, VERSION 2025.4-conda_forge (single precision)
Reading file /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_7d068895-4af2-4660-adfe-699247e

2026-07-24 11:42:54,146 [MainThread  ] [INFO ]  Removed: ['/home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_7d068895-4af2-4660-adfe-699247ead240', '/home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/468e3b7e-83ff-4f24-8a90-39c843898340.stdin']
2026-07-24 11:42:54,147 [MainThread  ] [INFO ]  


0

In [44]:
import plotly.graph_objs as go

# Read Rgyr data from file
with open(output_rgyr, 'r') as rgyr_file:
    x, y = zip(*[
        (float(line.split()[0]), float(line.split()[1]))
        for line in rgyr_file
        if not line.startswith(("#", "@"))
    ])

# Create a scatter plot
fig = go.Figure(data=go.Scatter(x=x, y=y, mode='lines+markers'))

# Update layout
fig.update_layout(title="Radius of Gyration",
                  xaxis_title="Time (ps)",
                  yaxis_title="Rgyr (nm)",
                  height=600)

# Show the figure
fig.show()

<a id="post"></a>
***
## Post-processing and Visualizing resulting 3D trajectory
Post-processing and Visualizing the **protein system** MD setup **resulting trajectory** using **NGL**
- [Step 1](#ppStep1): *Imaging* the resulting trajectory, **stripping out water molecules and ions** and **correcting periodicity issues**.
- [Step 2](#ppStep2): Generating a *dry* structure, **removing water molecules and ions** from the final snapshot of the MD setup pipeline.
- [Step 3](#ppStep3): Visualizing the *imaged* trajectory using the *dry* structure as a **topology**. 
***
**Building Blocks** used:
 - [GMXImage](https://biobb-analysis.readthedocs.io/en/latest/gromacs.html#module-gromacs.gmx_image) from **biobb_analysis.gromacs.gmx_image** 
 - [GMXTrjConvStr](https://biobb-analysis.readthedocs.io/en/latest/gromacs.html#module-gromacs.gmx_trjconv_str) from **biobb_analysis.gromacs.gmx_trjconv_str** 
***

<a id="ppStep1"></a>
### Step 1: *Imaging* the resulting trajectory.
Stripping out **water molecules and ions** and **correcting periodicity issues**  

In [45]:
# GMXImage: "Imaging" the resulting trajectory
#           Removing water molecules and ions from the resulting structure
from biobb_analysis.gromacs.gmx_image import gmx_image

# Create prop dict and inputs/outputs
output_imaged_traj = 'MD_sym/peptide_imaged_traj.trr'
prop = {
    'center_selection':  'Protein',
    'output_selection': 'Protein',
    'pbc' : 'mol',
    'center' : True
}

# Create and launch bb
gmx_image(input_traj_path=output_md_trr,
         input_top_path=output_gppmd_tpr,
         output_traj_path=output_imaged_traj, 
          properties=prop)

2026-07-24 11:42:55,821 [MainThread  ] [INFO ]  Module: biobb_analysis.gromacs.gmx_image Version: 5.2.1
2026-07-24 11:42:55,821 [MainThread  ] [INFO ]  Directory successfully created: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_5d05d641-5f21-4973-9d2a-969bed6aa17f
2026-07-24 11:42:55,821 [MainThread  ] [INFO ]  Copy to stage: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/MD_sym/peptide_md.trr --> sandbox_5d05d641-5f21-4973-9d2a-969bed6aa17f
2026-07-24 11:42:55,844 [MainThread  ] [INFO ]  Copy to stage: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/MD_sym/peptide_gppmd.tpr --> sandbox_5d05d641-5f21-4973-9d2a-969bed6aa17f
2026-07-24 11:42:55,845 [MainThread  ] [INFO ]  Copy to stage: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/81302909-c110-43e0-880d-de2f92e74e44.stdin --> sandbox_5d05d641-5f21-4973-9d2a-969bed6aa17f
2026-07-24 11:42:55,846 [MainThread  ] [INFO ]  Launching command (it may take a while): gmx trjconv -f /home/laf

2026-07-24 11:42:56,221 [MainThread  ] [INFO ]                 :-) GROMACS - gmx trjconv, 2025.4-conda_forge (-:

Executable:   /home/lafayette/miniconda3/envs/MD/bin.AVX2_256/gmx
Data prefix:  /home/lafayette/miniconda3/envs/MD
Working dir:  /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide
Command line:
  gmx trjconv -f /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_5d05d641-5f21-4973-9d2a-969bed6aa17f/peptide_md.trr -s /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_5d05d641-5f21-4973-9d2a-969bed6aa17f/peptide_gppmd.tpr -fit none -o /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_5d05d641-5f21-4973-9d2a-969bed6aa17f/peptide_imaged_traj.trr -center -pbc mol -ur compact

Will write trr: Trajectory in portable xdr format
Reading file /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_5d05d641-5f21-4973-9d2a-969bed6aa17f/peptide_gppmd.tpr, VERSION 2025.4-conda_forge (single precision)
Reading file /home/lafaye

2026-07-24 11:42:56,226 [MainThread  ] [INFO ]  Removed: ['/home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_5d05d641-5f21-4973-9d2a-969bed6aa17f', '/home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/81302909-c110-43e0-880d-de2f92e74e44.stdin']
2026-07-24 11:42:56,226 [MainThread  ] [INFO ]  


0

<a id="ppStep2"></a>
### Step 2: Generating the output *dry* structure.
**Removing water molecules and ions** from the resulting structure

In [ ]:
# GMXTrjConvStr: Converting and/or manipulating a structure
#                Removing water molecules and ions from the resulting structure
#                The "dry" structure will be used as a topology to visualize 
#                the "imaged dry" trajectory generated in the previous step.
from biobb_analysis.gromacs.gmx_trjconv_str import gmx_trjconv_str

# Create prop dict and inputs/outputs
output_dry_gro = 'MD_sym/peptide_md_dry.gro'
prop = {
    'selection':  'Protein'
}

# Create and launch bb
gmx_trjconv_str(input_structure_path=output_md_gro,
         input_top_path=output_gppmd_tpr,
         output_str_path=output_dry_gro, 
          properties=prop)

2026-07-24 11:43:13,289 [MainThread  ] [INFO ]  Module: biobb_analysis.gromacs.gmx_trjconv_str Version: 5.2.1
2026-07-24 11:43:13,290 [MainThread  ] [INFO ]  Directory successfully created: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_21536615-4ea2-4f45-b762-bdd4a7e05d80
2026-07-24 11:43:13,290 [MainThread  ] [INFO ]  Copy to stage: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/MD_sym/peptide_md.gro --> sandbox_21536615-4ea2-4f45-b762-bdd4a7e05d80
2026-07-24 11:43:13,291 [MainThread  ] [INFO ]  Copy to stage: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/MD_sym/peptide_gppmd.tpr --> sandbox_21536615-4ea2-4f45-b762-bdd4a7e05d80
2026-07-24 11:43:13,291 [MainThread  ] [INFO ]  Copy to stage: /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/09676e99-ec0c-4f8f-b2b4-9465bc786681.stdin --> sandbox_21536615-4ea2-4f45-b762-bdd4a7e05d80
2026-07-24 11:43:13,292 [MainThread  ] [INFO ]  Launching command (it may take a while): gmx trjconv -f /ho

2026-07-24 11:43:13,307 [MainThread  ] [INFO ]                 :-) GROMACS - gmx trjconv, 2025.4-conda_forge (-:

Executable:   /home/lafayette/miniconda3/envs/MD/bin.AVX2_256/gmx
Data prefix:  /home/lafayette/miniconda3/envs/MD
Working dir:  /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide
Command line:
  gmx trjconv -f /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_21536615-4ea2-4f45-b762-bdd4a7e05d80/peptide_md.gro -s /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_21536615-4ea2-4f45-b762-bdd4a7e05d80/peptide_gppmd.tpr -o /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_21536615-4ea2-4f45-b762-bdd4a7e05d80/peptide_md_dry.gro -nocenter

Will write gro: Coordinate file in Gromos-87 format
Reading file /home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_21536615-4ea2-4f45-b762-bdd4a7e05d80/peptide_gppmd.tpr, VERSION 2025.4-conda_forge (single precision)
Reading file /home/lafayette/Escritorio/PeptideCrowding/M

2026-07-24 11:43:13,308 [MainThread  ] [INFO ]  Removed: ['/home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/sandbox_21536615-4ea2-4f45-b762-bdd4a7e05d80', '/home/lafayette/Escritorio/PeptideCrowding/MD_LonePeptide/09676e99-ec0c-4f8f-b2b4-9465bc786681.stdin']
2026-07-24 11:43:13,309 [MainThread  ] [INFO ]  


0

<a id="ppStep3"></a>
### Step 3: Visualizing the generated dehydrated trajectory.
Using the **imaged trajectory** (output of the [Post-processing step 1](#ppStep1)) with the **dry structure** (output of the [Post-processing step 2](#ppStep2)) as a topology.

In [47]:
# NOTE: don't execute this cell when running in google colab, as it is not compatible with simpletraj

# # Show trajectory
# view = nglview.show_simpletraj(nglview.SimpletrajTrajectory(output_imaged_traj, output_dry_gro), gui=True)
# view

<a id="output"></a>
## Output files

Important **Output files** generated:
 - **output_md_gro** (1aki_md.gro): **Final structure** (snapshot) of the MD setup protocol.
 - **output_md_trr** (1aki_md.trr): **Final trajectory** of the MD setup protocol.
 - **output_md_cpt** (1aki_md.cpt): **Final checkpoint file**, with information about the state of the simulation. It can be used to **restart** or **continue** a MD simulation.
 - **output_gppmd_tpr** (1aki_gppmd.tpr): **Final tpr file**, GROMACS portable binary run input file. This file contains the starting structure of the **MD setup free MD simulation step**, together with the molecular topology and all the simulation parameters. It can be used to **extend** the simulation.
 - **output_genion_top_zip** (1aki_genion_top.zip): **Final topology** of the MD system. It is a compressed zip file including a **topology file** (.top) and a set of auxiliary **include topology** files (.itp).

**Analysis** (MD setup check) output files generated:
 - **output_rms_first** (1aki_rms_first.xvg): **Root Mean Square deviation (RMSd)** against **minimized and equilibrated structure** of the final **free MD run step**.
 - **output_rms_exp** (1aki_rms_exp.xvg): **Root Mean Square deviation (RMSd)** against **experimental structure** of the final **free MD run step**.
 - **output_rgyr** (1aki_rgyr.xvg): **Radius of Gyration** of the final **free MD run step** of the **setup pipeline**.
 

***
<a id="questions"></a>

## Questions & Comments

Questions, issues, suggestions and comments are really welcome!

* GitHub issues:
    * [https://github.com/bioexcel/biobb](https://github.com/bioexcel/biobb)

* BioExcel forum:
    * [https://ask.bioexcel.eu/c/BioExcel-Building-Blocks-library](https://ask.bioexcel.eu/c/BioExcel-Building-Blocks-library)
